# Contrastive Loss Comparison — Analysis Notebook

Структура:
- **[0] Setup** — импорты и хелперы
- **[1] Данные** — загрузка/подготовка датасетов
- **[2–6] Эксперименты** — по одному блоку на эксперимент (запуск + результаты)
- **[7] Сравнение** — итоговые таблицы и графики по всем экспериментам

---
## [0] Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # без GUI
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from IPython.display import display, Image

from config import load_config, get_run_dir
from training.trainer import train_one_run
from training.train_utils import get_device

sns.set_theme(style='whitegrid', palette='tab10')
%config InlineBackend.figure_format = 'retina'
matplotlib.use('inline')  # переключаемся на inline после импортов

COLORS = {'info_nce': '#1f77b4', 'triplet': '#ff7f0e', 'cosine': '#2ca02c'}
LOSS_LABELS = {'info_nce': 'InfoNCE', 'triplet': 'Triplet', 'cosine': 'Cosine'}
MODE_LABELS = {'nli_only': 'NLI only', 'sts_only': 'STS only', 'nli_plus_sts': 'NLI + STS'}
OUTPUT_DIR = '../outputs'

print(f'Device: {get_device()}')

In [ ]:
# ── Хелперы ──────────────────────────────────────────────────────────────────

def run_experiment(config_path, lr, max_train_samples=None, skip_if_exists=True):
    """Запустить один эксперимент. Если уже выполнен — загрузить результат."""
    cfg = load_config(config_path)
    if max_train_samples:
        cfg.training.max_train_samples = max_train_samples
    cfg.output_dir = OUTPUT_DIR

    run_dir = get_run_dir(OUTPUT_DIR, cfg.loss.type, lr, cfg.training_mode, cfg.model_name)
    result_path = os.path.join(run_dir, 'metrics', 'result.json')

    if skip_if_exists and os.path.exists(result_path):
        print(f'✓ Уже выполнен, загружаю результат: {result_path}')
        with open(result_path) as f:
            result = json.load(f)
        result['_run_dir'] = run_dir
        result['_skipped'] = True
        return result

    print(f'▶ Запуск: {cfg.loss.type} | {cfg.training_mode} | lr={lr}')
    t0 = time.time()
    result = train_one_run(cfg, lr=lr, run_dir=run_dir)
    result['_run_dir'] = run_dir
    result['_skipped'] = False

    with open(result_path, 'w') as f:
        json.dump({k: v for k, v in result.items() if not k.startswith('_')}, f, indent=2)

    print(f'Готово за {time.time() - t0:.1f}с')
    return result


def load_logs(run_dir):
    train_log = os.path.join(run_dir, 'logs', 'train_log.csv')
    val_log   = os.path.join(run_dir, 'logs', 'val_log.csv')
    train_df = pd.read_csv(train_log) if os.path.exists(train_log) else pd.DataFrame()
    val_df   = pd.read_csv(val_log)   if os.path.exists(val_log)   else pd.DataFrame()
    return train_df, val_df


def show_metrics(result):
    """Вывести ключевые метрики одного эксперимента."""
    loss_label = LOSS_LABELS.get(result.get('loss_type', ''), result.get('loss_type', ''))
    mode_label = MODE_LABELS.get(result.get('training_mode', ''), result.get('training_mode', ''))

    print('─' * 50)
    print(f'  Loss:               {loss_label}')
    print(f'  Mode:               {mode_label}')
    print(f'  Learning rate:      {result.get("learning_rate", "-"):.0e}')
    print(f'  Spearman (val):     {result.get("best_spearman_val", 0):.4f}')
    print(f'  Spearman (test):    {result.get("best_spearman_test", 0):.4f}')
    print(f'  Training time:      {result.get("training_time", 0):.1f}s')
    print(f'  Epochs run:         {result.get("epochs_run", "-")}')
    if result.get('diverged'):
        print('  ⚠ DIVERGED (NaN/Inf loss)')
    print('─' * 50)


def plot_run(result, figsize=(13, 4)):
    """Нарисовать train loss и Spearman validation для одного запуска."""
    run_dir = result.get('_run_dir')
    if not run_dir:
        print('run_dir не найден')
        return

    train_df, val_df = load_logs(run_dir)
    loss_type  = result.get('loss_type', '')
    mode_label = MODE_LABELS.get(result.get('training_mode', ''), '')
    lr         = result.get('learning_rate', '')
    color      = COLORS.get(loss_type, 'steelblue')
    title_base = f"{LOSS_LABELS.get(loss_type, loss_type)} | {mode_label} | lr={lr:.0e}"

    n_cols = 3 if (not val_df.empty and 'epoch_time_s' in val_df.columns) else 2
    fig, axes = plt.subplots(1, n_cols, figsize=figsize)

    # Train loss
    if not train_df.empty and 'step' in train_df.columns:
        axes[0].plot(train_df['step'], train_df['train_loss'], color=color, linewidth=1.8)
        axes[0].set_title(f'Train Loss\n{title_base}')
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('Loss')
    else:
        axes[0].text(0.5, 0.5, 'нет данных', ha='center', va='center', transform=axes[0].transAxes)

    # Spearman validation
    if not val_df.empty and 'spearman_score' in val_df.columns:
        # Разные стадии разным цветом
        if 'stage' in val_df.columns:
            stage_colors = {'nli': color, 'sts': '#e377c2', 'sts_finetune': '#bcbd22'}
            for stage, grp in val_df.groupby('stage'):
                axes[1].plot(grp['epoch'], grp['spearman_score'], marker='o', linewidth=1.8,
                             label=stage, color=stage_colors.get(stage, 'gray'))
            axes[1].legend(fontsize=8)
        else:
            axes[1].plot(val_df['epoch'], val_df['spearman_score'], marker='o',
                         color=color, linewidth=1.8)
        axes[1].set_title(f'Spearman (val)\n{title_base}')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Spearman')
        best = result.get('best_spearman_val', 0)
        axes[1].axhline(best, color='red', linestyle='--', linewidth=1, alpha=0.6,
                        label=f'best={best:.4f}')
        axes[1].legend(fontsize=8)

    # Grad norm
    if n_cols == 3 and not train_df.empty and 'grad_norm' in train_df.columns:
        axes[2].plot(train_df['step'], train_df['grad_norm'], color='gray', linewidth=1.2, alpha=0.8)
        axes[2].set_title(f'Gradient Norm\n{title_base}')
        axes[2].set_xlabel('Step')
        axes[2].set_ylabel('Grad Norm')

    plt.tight_layout()
    plt.show()


def load_all_results(output_dir):
    rows = []
    for root, _, files in os.walk(output_dir):
        if 'result.json' in files:
            try:
                with open(os.path.join(root, 'result.json')) as f:
                    r = json.load(f)
                    r['_run_dir'] = root
                    rows.append(r)
            except Exception:
                pass
    return pd.DataFrame(rows) if rows else pd.DataFrame()


print('Хелперы загружены.')

---
## [1] Подготовка данных

Запустить один раз перед первым экспериментом.

In [ ]:
# Поставить max_samples=None для полного датасета
MAX_SAMPLES = 10_000  # уменьшить для быстрого теста

from datasets.prepare_nli import prepare_and_save
from datasets.prepare_sts import prepare_sts_and_save

prepare_and_save(processed_dir='../data/processed', cache_dir='../data/cache',
                 max_samples=MAX_SAMPLES)
prepare_sts_and_save(processed_dir='../data/processed', cache_dir='../data/cache')

print('Данные готовы.')

---
## [2] Эксперимент 1 — InfoNCE · NLI only

Contrastive learning с in-batch negatives, temperature=0.05.

In [ ]:
result_info_nce = run_experiment(
    config_path='../experiments/info_nce_nli.yaml',
    lr=2e-5,
    max_train_samples=MAX_SAMPLES,
    skip_if_exists=True,
)

In [ ]:
show_metrics(result_info_nce)
plot_run(result_info_nce)

---
## [3] Эксперимент 2 — Triplet · NLI only

Metric learning: d(a,n) > d(a,p) + margin, cosine distance, margin=0.3.

In [ ]:
result_triplet = run_experiment(
    config_path='../experiments/triplet_nli.yaml',
    lr=2e-5,
    max_train_samples=MAX_SAMPLES,
    skip_if_exists=True,
)

In [ ]:
show_metrics(result_triplet)
plot_run(result_triplet)

---
## [4] Эксперимент 3 — Cosine · NLI only

Regression: MSE(cosine_sim, label_score), entailment=1.0, neutral=0.5, contradiction=0.0.

In [ ]:
result_cosine_nli = run_experiment(
    config_path='../experiments/cosine_nli.yaml',
    lr=2e-5,
    max_train_samples=MAX_SAMPLES,
    skip_if_exists=True,
)

In [ ]:
show_metrics(result_cosine_nli)
plot_run(result_cosine_nli)

---
## [5] Эксперимент 4 — Cosine · STS only

Human similarity supervision: MSE(cosine_sim, normalized_human_score).

In [ ]:
result_cosine_sts = run_experiment(
    config_path='../experiments/cosine_sts.yaml',
    lr=2e-5,
    max_train_samples=None,  # STS train маленький (~5749 пар), берём всё
    skip_if_exists=True,
)

In [ ]:
show_metrics(result_cosine_sts)
plot_run(result_cosine_sts)

---
## [6] Эксперимент 5 — Cosine · NLI + STS (двухэтапное)

Stage 1: NLI (entailment supervision) → Stage 2: STS fine-tuning (human annotations).

In [ ]:
result_cosine_ft = run_experiment(
    config_path='../experiments/cosine_nli_plus_sts.yaml',
    lr=2e-5,
    max_train_samples=MAX_SAMPLES,
    skip_if_exists=True,
)

In [ ]:
show_metrics(result_cosine_ft)
plot_run(result_cosine_ft)

# Дельта от STS fine-tuning
if result_cosine_ft.get('spearman_before_sts_ft') is not None:
    before = result_cosine_ft['spearman_before_sts_ft']
    after  = result_cosine_ft.get('best_spearman_test', 0)
    delta  = after - before
    sign   = '↑' if delta >= 0 else '↓'
    print(f'STS fine-tuning: {before:.4f} → {after:.4f}  ({sign}{abs(delta):.4f})')

---
## [7] Сравнение всех экспериментов

Ячейки ниже загружают все `result.json` из `outputs/` и строят итоговые таблицы и графики.

In [ ]:
df = load_all_results(OUTPUT_DIR)
print(f'Загружено результатов: {len(df)}')
col = 'best_spearman_test' if 'best_spearman_test' in df.columns else 'best_spearman'
display(df[['loss_type', 'training_mode', 'learning_rate', col, 'training_time', 'epochs_run']]
        .sort_values(col, ascending=False)
        .reset_index(drop=True))

In [ ]:
# ── Таблица 1: лучший результат по каждой loss function (NLI-only) ─────────
nli_df = df[df['training_mode'] == 'nli_only'] if 'training_mode' in df.columns else df

best = (
    nli_df.sort_values(col, ascending=False)
    .groupby('loss_type').first().reset_index()
)
stability = nli_df.groupby('loss_type')[col].agg(mean='mean', std='std').reset_index()
merged = best.merge(stability, on='loss_type')
merged['Loss'] = merged['loss_type'].map(LOSS_LABELS)
merged['Best LR'] = merged['learning_rate'].apply(lambda x: f'{x:.0e}')
merged['Best Spearman'] = merged[col].apply(lambda x: f'{x:.4f}')
merged['Mean ± Std'] = merged.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
merged['Time'] = merged['training_time'].apply(lambda x: f'{x:.0f}s')

print('=== Best Result per Loss Function (NLI-only) ===')
display(merged[['Loss', 'Best LR', 'Best Spearman', 'Mean ± Std', 'Time']])

In [ ]:
# ── Таблица 2: сравнение supervision strategies ────────────────────────────
if 'training_mode' in df.columns:
    sup = (
        df.sort_values(col, ascending=False)
        .groupby(['training_mode', 'loss_type']).first().reset_index()
    )
    sup['Mode'] = sup['training_mode'].map(MODE_LABELS)
    sup['Loss'] = sup['loss_type'].map(LOSS_LABELS)
    sup['Best Spearman'] = sup[col].apply(lambda x: f'{x:.4f}')
    sup['Best LR'] = sup['learning_rate'].apply(lambda x: f'{x:.0e}')
    print('=== Supervision Strategy Comparison ===')
    display(sup[['Mode', 'Loss', 'Best LR', 'Best Spearman']].sort_values('Mode'))

In [ ]:
# ── График: Train Loss сравнение (лучший LR) ──────────────────────────────
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

fig, ax = plt.subplots(figsize=(10, 5))

run_results = [r for r in [result_info_nce, result_triplet, result_cosine_nli]
               if r is not None and '_run_dir' in r]

for r in run_results:
    train_df, _ = load_logs(r['_run_dir'])
    if not train_df.empty and 'step' in train_df.columns:
        lt = r.get('loss_type', '')
        ax.plot(train_df['step'], train_df['train_loss'],
                label=LOSS_LABELS.get(lt, lt), color=COLORS.get(lt), linewidth=2)

ax.set_xlabel('Step')
ax.set_ylabel('Train Loss')
ax.set_title('Train Loss — Сравнение методов (NLI only, lr=2e-5)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── График: Spearman validation сравнение ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for r in run_results:
    _, val_df = load_logs(r['_run_dir'])
    if not val_df.empty and 'spearman_score' in val_df.columns:
        lt = r.get('loss_type', '')
        # только NLI-стадия
        vdf = val_df[val_df['stage'] == 'nli'] if 'stage' in val_df.columns else val_df
        ax.plot(vdf['epoch'], vdf['spearman_score'], marker='o',
                label=LOSS_LABELS.get(lt, lt), color=COLORS.get(lt), linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Spearman (val)')
ax.set_title('Validation Spearman — Сравнение методов (NLI only, lr=2e-5)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── График: Spearman vs LR ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for lt in nli_df['loss_type'].unique():
    sub = nli_df[nli_df['loss_type'] == lt].sort_values('learning_rate')
    ax.plot(sub['learning_rate'].astype(str), sub[col], marker='o',
            label=LOSS_LABELS.get(lt, lt), color=COLORS.get(lt), linewidth=2)

ax.set_xlabel('Learning Rate')
ax.set_ylabel('Best Spearman (test)')
ax.set_title('Spearman vs Learning Rate (LR sensitivity)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Stability Heatmap: loss × LR ──────────────────────────────────────────
if len(nli_df) > 1:
    pivot = nli_df.pivot_table(index='loss_type', columns='learning_rate', values=col, aggfunc='max')
    pivot.index = [LOSS_LABELS.get(i, i) for i in pivot.index]
    pivot.columns = [f'{c:.0e}' for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(8, 3))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax, linewidths=0.5)
    ax.set_title('Spearman (test) — Loss × LR')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Convergence curves: одна модель, все LR на одном графике ──────────────
# Для каждой loss function — один рисунок: train loss (лево) и Spearman val (право)
# Каждая линия = один LR. Требует запущенных run_all_experiments.py (все 4 LR).

def load_all_lr_logs(output_dir, loss_type, training_mode='nli_only'):
    """Собрать train/val логи всех LR-запусков для одной (loss, mode) пары."""
    lr_logs = {}
    for root, _, files in os.walk(output_dir):
        if 'result.json' not in files:
            continue
        try:
            with open(os.path.join(root, 'result.json')) as f:
                r = json.load(f)
        except Exception:
            continue
        if r.get('loss_type') != loss_type:
            continue
        if r.get('training_mode', 'nli_only') != training_mode:
            continue
        lr_val = r.get('learning_rate')
        logs_dir = os.path.join(root, '..', 'logs')
        train_path = os.path.join(logs_dir, 'train_log.csv')
        val_path   = os.path.join(logs_dir, 'val_log.csv')
        train_df = pd.read_csv(train_path) if os.path.exists(train_path) else pd.DataFrame()
        val_df   = pd.read_csv(val_path)   if os.path.exists(val_path)   else pd.DataFrame()
        lr_logs[lr_val] = {'train': train_df, 'val': val_df}
    return lr_logs


lr_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for loss_type in ['info_nce', 'triplet', 'cosine']:
    lr_logs = load_all_lr_logs(OUTPUT_DIR, loss_type, training_mode='nli_only')
    if not lr_logs:
        print(f'{LOSS_LABELS[loss_type]}: нет данных (запустите run_all_experiments.py для всех LR)')
        continue

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'{LOSS_LABELS[loss_type]} — сходимость при разных LR', fontsize=13)

    for i, (lr, logs) in enumerate(sorted(lr_logs.items())):
        label = f'lr={lr:.0e}'
        color = lr_palette[i % len(lr_palette)]

        train_df = logs['train']
        if not train_df.empty and 'step' in train_df.columns:
            axes[0].plot(train_df['step'], train_df['train_loss'],
                         label=label, color=color, linewidth=1.8)

        val_df = logs['val']
        if not val_df.empty and 'spearman_score' in val_df.columns:
            vdf = val_df[val_df['stage'] == 'nli'] if 'stage' in val_df.columns else val_df
            axes[1].plot(vdf['epoch'], vdf['spearman_score'],
                         marker='o', label=label, color=color, linewidth=1.8)

    axes[0].set_xlabel('Step');  axes[0].set_ylabel('Train Loss');  axes[0].set_title('Train Loss');  axes[0].legend(fontsize=9)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Spearman (val)'); axes[1].set_title('Spearman (val)'); axes[1].legend(fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── График: Supervision strategy comparison ────────────────────────────────
if 'training_mode' in df.columns and df['training_mode'].nunique() > 1:
    best_per_mode = (
        df.sort_values(col, ascending=False)
        .groupby(['training_mode', 'loss_type']).first().reset_index()
    )

    modes = [m for m in ['nli_only', 'sts_only', 'nli_plus_sts']
             if m in best_per_mode['training_mode'].values]
    loss_types = sorted(best_per_mode['loss_type'].unique())
    x = np.arange(len(loss_types))
    width = 0.25
    mode_palette = ['#1f77b4', '#ff7f0e', '#2ca02c']

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, mode in enumerate(modes):
        sub = best_per_mode[best_per_mode['training_mode'] == mode]
        heights = [float(sub[sub['loss_type'] == lt][col].values[0])
                   if not sub[sub['loss_type'] == lt].empty else 0.0
                   for lt in loss_types]
        bars = ax.bar(x + i * width, heights, width, label=MODE_LABELS.get(mode, mode),
                      color=mode_palette[i], alpha=0.85)
        for bar, h in zip(bars, heights):
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.002,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x + width * (len(modes) - 1) / 2)
    ax.set_xticklabels([LOSS_LABELS.get(lt, lt) for lt in loss_types])
    ax.set_ylabel('Best Spearman (test)')
    ax.set_title('Supervision Strategy Comparison')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── График: Training Time vs Quality ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for lt in df['loss_type'].unique():
    sub = df[df['loss_type'] == lt]
    ax.scatter(sub['training_time'] / 60, sub[col],
               label=LOSS_LABELS.get(lt, lt), color=COLORS.get(lt), s=80, alpha=0.8)

ax.set_xlabel('Training Time (min)')
ax.set_ylabel('Best Spearman (test)')
ax.set_title('Training Time vs Quality')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Sensitivity analysis ───────────────────────────────────────────────────
print('=== LR Sensitivity ===')
sens = nli_df.groupby('loss_type')[col].agg(
    best='max', worst='min', mean='mean', std='std',
    range=lambda x: x.max() - x.min()
).reset_index()
sens['Loss'] = sens['loss_type'].map(LOSS_LABELS)
sens = sens[['Loss', 'best', 'worst', 'std', 'range']].rename(columns={
    'best': 'Best', 'worst': 'Worst', 'std': 'Std', 'range': 'Range (max-min)'
})
display(sens)

if not sens.empty:
    most_sensitive = sens.loc[sens['Std'].idxmax(), 'Loss']
    most_robust    = sens.loc[sens['Std'].idxmin(), 'Loss']
    print(f'  Наиболее чувствительна к LR: {most_sensitive}')
    print(f'  Наиболее устойчива к LR:     {most_robust}')

In [ ]:
# ── STS Fine-tuning Effect ─────────────────────────────────────────────────
if 'spearman_before_sts_ft' in df.columns:
    ft_df = df[df['spearman_before_sts_ft'].notna()].copy()
    if not ft_df.empty:
        ft_df['delta'] = ft_df[col] - ft_df['spearman_before_sts_ft']
        print('=== STS Fine-tuning Effect ===')
        display(ft_df[['loss_type', 'learning_rate', 'spearman_before_sts_ft', col, 'delta']])

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(ft_df['learning_rate'].astype(str), ft_df['delta'],
               color=['green' if d >= 0 else 'red' for d in ft_df['delta']], alpha=0.8)
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_xlabel('Learning Rate')
        ax.set_ylabel('Δ Spearman')
        ax.set_title('Δ Spearman: после STS fine-tuning − до')
        plt.tight_layout()
        plt.show()

In [ ]:
# ── Итоговый вывод ─────────────────────────────────────────────────────────
if not df.empty:
    best_row = df.loc[df[col].idxmax()]
    print('=== Итоговый вывод ===')
    print(f'  Лучший результат: {LOSS_LABELS.get(best_row["loss_type"], best_row["loss_type"])}')
    if 'training_mode' in best_row:
        print(f'  Стратегия:        {MODE_LABELS.get(best_row["training_mode"], best_row["training_mode"])}')
    print(f'  Learning rate:    {best_row["learning_rate"]:.0e}')
    print(f'  Spearman (test):  {best_row[col]:.4f}')